# Time Series Analysis of Air Raid Alerts in Ukraine

####This notebook is part of the KSE AI Agentic Summer School Stage 2 selection.


In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/Vadimkin/ukrainian-air-raid-sirens-dataset/main/datasets/volunteer_data_en.csv"

df = pd.read_csv(url)

print("Number of rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nData types and non-null values:")
df.info()

Number of rows and columns: (101969, 4)

Column names:
['region', 'started_at', 'finished_at', 'naive']

First five rows:


,region,started_at,finished_at,naive
0,Kyiv City,2022-02-25 16:36:22+00:00,2022-02-25 17:06:22+00:00,True
1,Cherkaska oblast,2022-02-25 18:36:21+00:00,2022-02-25 19:32:11+00:00,False
2,Rivnenska oblast,2022-02-25 18:56:44+00:00,2022-02-25 19:26:44+00:00,True
3,Zaporizka oblast,2022-02-25 18:57:51+00:00,2022-02-25 19:27:51+00:00,True
4,Volynska oblast,2022-02-25 19:41:57+00:00,2022-02-26 04:01:55+00:00,False



Data types and non-null values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101969 entries, 0 to 101968
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   region       101969 non-null  object
 1   started_at   101969 non-null  object
 2   finished_at  101969 non-null  object
 3   naive        101969 non-null  bool  
dtypes: bool(1), object(3)
memory usage: 2.4+ MB


In [2]:
df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce", utc=True)
df["finished_at"] = pd.to_datetime(df["finished_at"], errors="coerce", utc=True)

print("Timezone of started_at:", df["started_at"].dt.tz)
print("Timezone of finished_at:", df["finished_at"].dt.tz)

print("\nEarliest alert start:", df["started_at"].min())
print("Latest alert finish:", df["finished_at"].max())

print("\nFailed started_at conversions:", df["started_at"].isna().sum())
print("Failed finished_at conversions:", df["finished_at"].isna().sum())

naive_count = df["naive"].sum()
naive_percentage = naive_count / len(df) * 100

print("\nNumber of naive=True records:", naive_count)
print("Percentage of naive=True records:", round(naive_percentage, 2), "%")

Timezone of started_at: UTC
Timezone of finished_at: UTC

Earliest alert start: 2022-02-25 16:36:22+00:00
Latest alert finish: 2026-06-25 00:19:46+00:00

Failed started_at conversions: 0
Failed finished_at conversions: 0

Number of naive=True records: 5023
Percentage of naive=True records: 4.93 %


In [3]:
inspection_df = df.copy()

inspection_df["duration_minutes"] = (
    inspection_df["finished_at"] - inspection_df["started_at"]
).dt.total_seconds() / 60

duplicate_count = df.duplicated().sum()
zero_duration_count = (inspection_df["duration_minutes"] == 0).sum()
negative_duration_count = (inspection_df["duration_minutes"] < 0).sum()

print("Exact duplicate rows:", duplicate_count)
print("Zero-duration alerts:", zero_duration_count)
print("Negative-duration alerts:", negative_duration_count)

print("\nDuration summary in minutes:")
print(inspection_df["duration_minutes"].describe())

columns_to_show = [
    "region",
    "started_at",
    "finished_at",
    "naive",
    "duration_minutes"
]

print("\nFive shortest alert records:")
display(
    inspection_df
    .sort_values("duration_minutes", ascending=True)
    [columns_to_show]
    .head(5)
)

print("\nTen longest alert records:")
display(
    inspection_df
    .sort_values("duration_minutes", ascending=False)
    [columns_to_show]
    .head(10)
)

naive_durations = inspection_df.loc[
    inspection_df["naive"],
    "duration_minutes"
]

naive_exactly_30_count = (naive_durations == 30).sum()
all_naive_exactly_30 = (naive_durations == 30).all()

print("\nTotal naive=True records:", len(naive_durations))
print("Naive=True records with exactly 30-minute duration:", naive_exactly_30_count)
print("Do all naive=True records last exactly 30 minutes?:", all_naive_exactly_30)

Exact duplicate rows: 0
Zero-duration alerts: 5
Negative-duration alerts: 0

Duration summary in minutes:
count    101969.000000
mean         50.187812
std          85.013431
min           0.000000
25%          20.283333
50%          31.233333
75%          57.800000
max       10695.900000
Name: duration_minutes, dtype: float64

Five shortest alert records:


,region,started_at,finished_at,naive,duration_minutes
85550,Dnipropetrovska oblast,2026-01-15 22:14:22+00:00,2026-01-15 22:14:22+00:00,False,0.0
92431,Kirovohradska oblast,2026-03-28 19:28:39+00:00,2026-03-28 19:28:39+00:00,False,0.0
90593,Dnipropetrovska oblast,2026-03-11 22:43:16+00:00,2026-03-11 22:43:16+00:00,False,0.0
90662,Dnipropetrovska oblast,2026-03-12 12:08:51+00:00,2026-03-12 12:08:51+00:00,False,0.0
79584,Dnipropetrovska oblast,2025-11-03 22:27:41+00:00,2025-11-03 22:27:41+00:00,False,0.0



Ten longest alert records:


,region,started_at,finished_at,naive,duration_minutes
450,Zaporizka oblast,2022-03-04 08:02:43+00:00,2022-03-11 18:18:37+00:00,False,10695.900000
95341,Chernivetska oblast,2026-04-17 16:55:46+00:00,2026-04-25 02:04:55+00:00,False,10629.150000
95041,Zakarpatska oblast,2026-04-17 16:52:42+00:00,2026-04-22 19:17:47+00:00,False,7345.083333
95042,Ternopilska oblast,2026-04-17 16:55:41+00:00,2026-04-22 17:55:46+00:00,False,7260.083333
95043,Khmelnytska oblast,2026-04-17 16:55:46+00:00,2026-04-22 04:53:45+00:00,False,6477.983333
87019,Lvivska oblast,2026-01-29 01:30:51+00:00,2026-02-01 20:10:22+00:00,False,5439.516667
63565,Chernivetska oblast,2025-03-03 09:45:47+00:00,2025-03-06 15:39:57+00:00,False,4674.166667
79988,Zakarpatska oblast,2025-11-07 23:07:47+00:00,2025-11-09 22:58:23+00:00,False,2870.600000
79989,Chernivetska oblast,2025-11-07 23:09:40+00:00,2025-11-09 22:59:13+00:00,False,2869.550000
75840,Volynska oblast,2025-09-14 06:07:07+00:00,2025-09-14 21:47:52+00:00,False,940.750000



Total naive=True records: 5023
Naive=True records with exactly 30-minute duration: 5023
Do all naive=True records last exactly 30 minutes?: True


In [4]:
data_with_duration = df.copy()

data_with_duration["duration_minutes"] = (
    data_with_duration["finished_at"] - data_with_duration["started_at"]
).dt.total_seconds() / 60

deduplicated_df = data_with_duration.drop_duplicates().copy()

occurrence_df = deduplicated_df.copy()


duration_df = deduplicated_df.loc[
    (~deduplicated_df["naive"])
    & (deduplicated_df["duration_minutes"] > 0)
].copy()

print("Original df rows:", len(df))
print("Rows after duplicate removal:", len(deduplicated_df))
print("Occurrence dataset rows:", len(occurrence_df))
print("Duration dataset rows:", len(duration_df))

print("\nNaive=True rows in occurrence dataset:",
      occurrence_df["naive"].sum())

print("Naive=True rows in duration dataset:",
      duration_df["naive"].sum())

print("\nNon-positive durations in occurrence dataset:",
      (occurrence_df["duration_minutes"] <= 0).sum())

print("Non-positive durations in duration dataset:",
      (duration_df["duration_minutes"] <= 0).sum())

Original df rows: 101969
Rows after duplicate removal: 101969
Occurrence dataset rows: 101969
Duration dataset rows: 96941

Naive=True rows in occurrence dataset: 5023
Naive=True rows in duration dataset: 0

Non-positive durations in occurrence dataset: 5
Non-positive durations in duration dataset: 0


In [5]:
import hashlib
import pandas as pd

snapshot_filename = "volunteer_data_en_snapshot_2026-06-25.csv"

df.to_csv(snapshot_filename, index=False)

with open(snapshot_filename, "rb") as file:
    sha256_checksum = hashlib.sha256(file.read()).hexdigest()

saved_df = pd.read_csv(snapshot_filename)

print("Saved snapshot:", snapshot_filename)
print("SHA-256 checksum:", sha256_checksum)
print("Original dataframe shape:", df.shape)
print("Saved snapshot shape:", saved_df.shape)
print("Shapes match:", saved_df.shape == df.shape)

Saved snapshot: volunteer_data_en_snapshot_2026-06-25.csv
SHA-256 checksum: 0354ac29bf8086fa8ca0e51c5bd63041bdb4d8671da93dc189a40cf684f62a38
Original dataframe shape: (101969, 4)
Saved snapshot shape: (101969, 4)
Shapes match: True


In [6]:
from google.colab import files

files.download(snapshot_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import hashlib
import pandas as pd

snapshot_filename = "volunteer_data_en_snapshot_2026-06-25.csv"
expected_shape = (101969, 4)
expected_checksum = "0354ac29bf8086fa8ca0e51c5bd63041bdb4d8671da93dc189a40cf684f62a38"

with open(snapshot_filename, "rb") as file:
    actual_checksum = hashlib.sha256(file.read()).hexdigest()

snapshot_df = pd.read_csv(snapshot_filename)

snapshot_df["started_at"] = pd.to_datetime(
    snapshot_df["started_at"],
    errors="coerce",
    utc=True
)

snapshot_df["finished_at"] = pd.to_datetime(
    snapshot_df["finished_at"],
    errors="coerce",
    utc=True
)

print("Loaded snapshot shape:", snapshot_df.shape)
print("Expected shape:", expected_shape)
print("Shape matches:", snapshot_df.shape == expected_shape)

print("\nActual SHA-256:", actual_checksum)
print("Expected SHA-256:", expected_checksum)
print("Checksum matches:", actual_checksum == expected_checksum)

print("\nstarted_at timezone:", snapshot_df["started_at"].dt.tz)
print("finished_at timezone:", snapshot_df["finished_at"].dt.tz)

print("\nFailed started_at conversions:", snapshot_df["started_at"].isna().sum())
print("Failed finished_at conversions:", snapshot_df["finished_at"].isna().sum())

Loaded snapshot shape: (101969, 4)
Expected shape: (101969, 4)
Shape matches: True

Actual SHA-256: 0354ac29bf8086fa8ca0e51c5bd63041bdb4d8671da93dc189a40cf684f62a38
Expected SHA-256: 0354ac29bf8086fa8ca0e51c5bd63041bdb4d8671da93dc189a40cf684f62a38
Checksum matches: True

started_at timezone: UTC
finished_at timezone: UTC

Failed started_at conversions: 0
Failed finished_at conversions: 0


In [8]:
region_check = snapshot_df.copy()

# Extract the UTC calendar date from each alert start timestamp
region_check["start_date"] = region_check["started_at"].dt.normalize()

region_summary = (
    region_check
    .groupby("region")
    .agg(
        total_alert_records=("region", "size"),
        earliest_alert_start=("started_at", "min"),
        latest_alert_start=("started_at", "max"),
        unique_alert_days=("start_date", "nunique"),
        earliest_start_date=("start_date", "min"),
        latest_start_date=("start_date", "max"),
        naive_records=("naive", "sum")
    )
    .reset_index()
)

region_summary["total_calendar_days"] = (
    region_summary["latest_start_date"]
    - region_summary["earliest_start_date"]
).dt.days + 1

region_summary["alert_day_percentage"] = (
    region_summary["unique_alert_days"]
    / region_summary["total_calendar_days"]
    * 100
)

region_summary["naive_percentage"] = (
    region_summary["naive_records"]
    / region_summary["total_alert_records"]
    * 100
)

region_summary = region_summary[
    [
        "region",
        "total_alert_records",
        "earliest_alert_start",
        "latest_alert_start",
        "unique_alert_days",
        "total_calendar_days",
        "alert_day_percentage",
        "naive_records",
        "naive_percentage"
    ]
].sort_values(
    "alert_day_percentage",
    ascending=False
).reset_index(drop=True)

region_summary["alert_day_percentage"] = region_summary[
    "alert_day_percentage"
].round(2)

region_summary["naive_percentage"] = region_summary[
    "naive_percentage"
].round(2)

display(region_summary)

,region,total_alert_records,earliest_alert_start,latest_alert_start,unique_alert_days,total_calendar_days,alert_day_percentage,naive_records,naive_percentage
0,Dnipropetrovska oblast,11801,2022-02-27 13:34:45+00:00,2026-06-24 21:28:01+00:00,1577,1579,99.87,369,3.13
1,Kharkivska oblast,11167,2022-02-26 05:19:53+00:00,2026-06-25 00:10:21+00:00,1578,1581,99.81,657,5.88
2,Zaporizka oblast,8338,2022-02-25 18:57:51+00:00,2026-06-24 19:07:08+00:00,1568,1581,99.18,195,2.34
3,Donetska oblast,8126,2022-02-27 12:49:52+00:00,2026-06-24 16:31:38+00:00,1550,1579,98.16,614,7.56
4,Mykolaivska oblast,6676,2022-02-26 15:33:49+00:00,2026-06-24 21:02:32+00:00,1539,1580,97.41,536,8.03
5,Sumska oblast,7758,2022-02-26 07:51:34+00:00,2026-06-25 00:00:17+00:00,1533,1581,96.96,772,9.95
6,Kirovohradska oblast,5703,2022-02-27 18:01:14+00:00,2026-06-24 18:26:42+00:00,1502,1579,95.12,90,1.58
7,Poltavska oblast,5817,2022-02-27 18:14:34+00:00,2026-06-24 21:58:19+00:00,1498,1579,94.87,87,1.50
8,Khersonska oblast,5797,2022-02-26 04:40:38+00:00,2026-06-24 19:47:04+00:00,1393,1580,88.16,976,16.84
9,Cherkaska oblast,3524,2022-02-25 18:36:21+00:00,2026-06-24 18:26:35+00:00,1352,1581,85.52,73,2.07


In [9]:
kyiv_region_check = snapshot_df.copy()

# Convert UTC alert starts to Kyiv local time
kyiv_region_check["started_at_kyiv"] = (
    kyiv_region_check["started_at"]
    .dt.tz_convert("Europe/Kyiv")
)

# Extract the Kyiv calendar date
kyiv_region_check["start_date_kyiv"] = (
    kyiv_region_check["started_at_kyiv"]
    .dt.normalize()
)

kyiv_region_summary = (
    kyiv_region_check
    .groupby("region")
    .agg(
        total_alert_records=("region", "size"),
        earliest_alert_start_kyiv=("started_at_kyiv", "min"),
        latest_alert_start_kyiv=("started_at_kyiv", "max"),
        positive_days=("start_date_kyiv", "nunique"),
        earliest_local_date=("start_date_kyiv", "min"),
        latest_local_date=("start_date_kyiv", "max"),
        naive_records=("naive", "sum")
    )
    .reset_index()
)

kyiv_region_summary["total_calendar_days"] = (
    kyiv_region_summary["latest_local_date"]
    - kyiv_region_summary["earliest_local_date"]
).dt.days + 1

kyiv_region_summary["negative_days"] = (
    kyiv_region_summary["total_calendar_days"]
    - kyiv_region_summary["positive_days"]
)

kyiv_region_summary["positive_day_percentage"] = (
    kyiv_region_summary["positive_days"]
    / kyiv_region_summary["total_calendar_days"]
    * 100
)

kyiv_region_summary["negative_day_percentage"] = (
    kyiv_region_summary["negative_days"]
    / kyiv_region_summary["total_calendar_days"]
    * 100
)

kyiv_region_summary["naive_percentage"] = (
    kyiv_region_summary["naive_records"]
    / kyiv_region_summary["total_alert_records"]
    * 100
)

# Smaller values mean the class balance is closer to 50/50
kyiv_region_summary["distance_from_50"] = (
    kyiv_region_summary["positive_day_percentage"] - 50
).abs()

kyiv_region_summary = (
    kyiv_region_summary[
        [
            "region",
            "total_alert_records",
            "earliest_alert_start_kyiv",
            "latest_alert_start_kyiv",
            "positive_days",
            "negative_days",
            "total_calendar_days",
            "positive_day_percentage",
            "negative_day_percentage",
            "naive_records",
            "naive_percentage",
            "distance_from_50"
        ]
    ]
    .sort_values(
        ["distance_from_50", "total_calendar_days"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

percentage_columns = [
    "positive_day_percentage",
    "negative_day_percentage",
    "naive_percentage",
    "distance_from_50"
]

kyiv_region_summary[percentage_columns] = (
    kyiv_region_summary[percentage_columns].round(2)
)

display(kyiv_region_summary)

,region,total_alert_records,earliest_alert_start_kyiv,latest_alert_start_kyiv,positive_days,negative_days,total_calendar_days,positive_day_percentage,negative_day_percentage,naive_records,naive_percentage,distance_from_50
0,Khmelnytska oblast,1265,2022-02-26 16:10:29+02:00,2026-06-23 23:14:59+03:00,801,777,1578,50.76,49.24,14,1.11,0.76
1,Volynska oblast,1790,2022-02-25 21:41:57+02:00,2026-06-24 01:27:53+03:00,821,759,1580,51.96,48.04,42,2.35,1.96
2,Lvivska oblast,1365,2022-02-26 06:26:17+02:00,2026-06-24 21:27:37+03:00,742,837,1579,46.99,53.01,31,2.27,3.01
3,Rivnenska oblast,1143,2022-02-25 20:56:44+02:00,2026-06-23 23:14:59+03:00,732,847,1579,46.36,53.64,13,1.14,3.64
4,Ternopilska oblast,976,2022-02-26 17:28:36+02:00,2026-06-23 23:16:10+03:00,655,923,1578,41.51,58.49,4,0.41,8.49
5,Ivano-Frankivska oblast,878,2022-02-27 19:28:20+02:00,2026-06-23 23:16:30+03:00,616,961,1577,39.06,60.94,22,2.51,10.94
6,Chernivetska oblast,864,2022-02-27 23:25:55+02:00,2026-06-23 23:15:00+03:00,613,964,1577,38.87,61.13,10,1.16,11.13
7,Vinnytska oblast,1850,2022-02-25 22:55:42+02:00,2026-06-24 21:25:39+03:00,1010,570,1580,63.92,36.08,32,1.73,13.92
8,Zakarpatska oblast,759,2022-03-12 05:33:46+02:00,2026-06-23 23:14:50+03:00,550,1014,1564,35.17,64.83,8,1.05,14.83
9,Zhytomyrska oblast,2013,2022-02-26 08:05:54+02:00,2026-06-23 23:15:07+03:00,1024,554,1578,64.89,35.11,41,2.04,14.89


In [10]:
selected_region = "Khmelnytska oblast"
start_date = "2022-02-26"
end_date = "2026-06-24"


region_df = snapshot_df.loc[
    snapshot_df["region"] == selected_region
].copy()


region_df["started_at_kyiv"] = (
    region_df["started_at"]
    .dt.tz_convert("Europe/Kyiv")
)


region_df["date"] = pd.to_datetime(
    region_df["started_at_kyiv"].dt.date
)


daily_alert_counts = (
    region_df
    .groupby("date")
    .size()
    .rename("alert_count")
)


complete_dates = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)


daily_df = (
    daily_alert_counts
    .reindex(complete_dates, fill_value=0)
    .rename_axis("date")
    .reset_index()
)


daily_df["alert_started_today"] = (
    daily_df["alert_count"] > 0
).astype(int)


duplicate_date_count = daily_df["date"].duplicated().sum()
dates_are_unique = daily_df["date"].is_unique

expected_number_of_days = len(complete_dates)
missing_date_count = expected_number_of_days - daily_df["date"].nunique()
missing_value_count = daily_df.isna().sum().sum()


total_days = len(daily_df)
positive_days = daily_df["alert_started_today"].sum()
negative_days = total_days - positive_days

positive_percentage = positive_days / total_days * 100
negative_percentage = negative_days / total_days * 100

print("Selected region:", selected_region)
print("Observation window:", start_date, "to", end_date)

print("\nTotal days:", total_days)
print("Expected days:", expected_number_of_days)

print("\nDuplicate dates:", duplicate_date_count)
print("Every date appears exactly once:", dates_are_unique)
print("Missing dates:", missing_date_count)
print("Missing values:", missing_value_count)

print("\nPositive days:", positive_days)
print("Negative days:", negative_days)
print("Positive-day percentage:", round(positive_percentage, 2), "%")
print("Negative-day percentage:", round(negative_percentage, 2), "%")

print("\nFirst 10 rows:")
display(daily_df.head(10))

print("\nLast 10 rows:")
display(daily_df.tail(10))

Selected region: Khmelnytska oblast
Observation window: 2022-02-26 to 2026-06-24

Total days: 1580
Expected days: 1580

Duplicate dates: 0
Every date appears exactly once: True
Missing dates: 0
Missing values: 0

Positive days: 801
Negative days: 779
Positive-day percentage: 50.7 %
Negative-day percentage: 49.3 %

First 10 rows:


,date,alert_count,alert_started_today
0,2022-02-26,1,1
1,2022-02-27,2,1
2,2022-02-28,3,1
3,2022-03-01,4,1
4,2022-03-02,1,1
5,2022-03-03,3,1
6,2022-03-04,1,1
7,2022-03-05,0,0
8,2022-03-06,4,1
9,2022-03-07,3,1



Last 10 rows:


,date,alert_count,alert_started_today
1570,2026-06-15,1,1
1571,2026-06-16,0,0
1572,2026-06-17,0,0
1573,2026-06-18,1,1
1574,2026-06-19,0,0
1575,2026-06-20,0,0
1576,2026-06-21,3,1
1577,2026-06-22,1,1
1578,2026-06-23,2,1
1579,2026-06-24,0,0


In [11]:
selected_region = "Khmelnytska oblast"
start_date = pd.Timestamp("2022-02-26")
end_date = pd.Timestamp("2026-06-24")

source_check_df = snapshot_df.loc[
    snapshot_df["region"] == selected_region
].copy()

source_check_df["local_date"] = pd.to_datetime(
    source_check_df["started_at"]
    .dt.tz_convert("Europe/Kyiv")
    .dt.date
)

source_records_in_window = source_check_df.loc[
    source_check_df["local_date"].between(start_date, end_date)
].shape[0]

daily_count_sum = daily_df["alert_count"].sum()

print("Source records inside window:", source_records_in_window)
print("Sum of daily alert_count:", daily_count_sum)
print("Aggregation counts match:", source_records_in_window == daily_count_sum)


model_df = daily_df.copy()


model_df["alert_started_tomorrow"] = (
    model_df["alert_started_today"].shift(-1)
)


shift_check = model_df.loc[
    0:9,
    ["date", "alert_started_today", "alert_started_tomorrow"]
].copy()


model_df = model_df.iloc[:-1].copy()

model_df["alert_started_tomorrow"] = (
    model_df["alert_started_tomorrow"].astype(int)
)


shift_is_correct = (
    model_df["alert_started_tomorrow"].to_numpy()
    == daily_df["alert_started_today"].iloc[1:].to_numpy()
).all()

print("\nTarget shift is correct:", shift_is_correct)
print("\nConsecutive rows used to inspect the shift:")
display(shift_check)

print("\nDuplicate dates:", model_df["date"].duplicated().sum())
print("Every date is unique:", model_df["date"].is_unique)
print("Missing values:", model_df.isna().sum().sum())


target_counts = (
    model_df["alert_started_tomorrow"]
    .value_counts()
    .sort_index()
)

target_percentages = (
    model_df["alert_started_tomorrow"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nFinal modelling rows:", len(model_df))
print("First modelling date:", model_df["date"].min())
print("Last modelling date:", model_df["date"].max())

print("\nTarget class counts:")
print(target_counts.rename({
    0: "No alert tomorrow",
    1: "Alert tomorrow"
}))

print("\nTarget class percentages:")
print(target_percentages.rename({
    0: "No alert tomorrow",
    1: "Alert tomorrow"
}))

Source records inside window: 1265
Sum of daily alert_count: 1265
Aggregation counts match: True

Target shift is correct: True

Consecutive rows used to inspect the shift:


,date,alert_started_today,alert_started_tomorrow
0,2022-02-26,1,1.0
1,2022-02-27,1,1.0
2,2022-02-28,1,1.0
3,2022-03-01,1,1.0
4,2022-03-02,1,1.0
5,2022-03-03,1,1.0
6,2022-03-04,1,0.0
7,2022-03-05,0,1.0
8,2022-03-06,1,1.0
9,2022-03-07,1,1.0



Duplicate dates: 0
Every date is unique: True
Missing values: 0

Final modelling rows: 1579
First modelling date: 2022-02-26 00:00:00
Last modelling date: 2026-06-23 00:00:00

Target class counts:
alert_started_tomorrow
No alert tomorrow    779
Alert tomorrow       800
Name: count, dtype: int64

Target class percentages:
alert_started_tomorrow
No alert tomorrow    49.34
Alert tomorrow       50.66
Name: proportion, dtype: float64


In [12]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

split_df = model_df.sort_values("date").reset_index(drop=True).copy()

dates_strictly_increasing = (
    split_df["date"].diff().dropna() > pd.Timedelta(0)
).all()

print("Dates are strictly increasing:", dates_strictly_increasing)
print("Total available observations:", len(split_df))


train_size = 1105
validation_size = 237
test_size = 237

train_df = split_df.iloc[:train_size].copy()

validation_df = split_df.iloc[
    train_size : train_size + validation_size
].copy()

test_df = split_df.iloc[
    train_size + validation_size :
].copy()


def print_period_summary(name, period_df):
    target_counts = (
        period_df["alert_started_tomorrow"]
        .value_counts()
        .sort_index()
        .reindex([0, 1], fill_value=0)
    )

    target_percentages = (
        target_counts / len(period_df) * 100
    ).round(2)

    print(f"\n{name}")
    print("Start date:", period_df["date"].min())
    print("End date:", period_df["date"].max())
    print("Rows:", len(period_df))

    print("No alert tomorrow:",
          target_counts[0],
          f"({target_percentages[0]}%)")

    print("Alert tomorrow:",
          target_counts[1],
          f"({target_percentages[1]}%)")


print_period_summary("Training period", train_df)
print_period_summary("Validation period", validation_df)
print_period_summary("Test period", test_df)


train_dates = set(train_df["date"])
validation_dates = set(validation_df["date"])
test_dates = set(test_df["date"])

no_train_validation_overlap = train_dates.isdisjoint(validation_dates)
no_train_test_overlap = train_dates.isdisjoint(test_dates)
no_validation_test_overlap = validation_dates.isdisjoint(test_dates)

all_rows_assigned = (
    len(train_df) + len(validation_df) + len(test_df)
    == len(split_df)
    == 1579
)

all_dates_assigned_once = (
    pd.concat([
        train_df[["date"]],
        validation_df[["date"]],
        test_df[["date"]]
    ])["date"].nunique()
    == 1579
)

print("\nSplit verification:")
print("Train and validation do not overlap:",
      no_train_validation_overlap)
print("Train and test do not overlap:",
      no_train_test_overlap)
print("Validation and test do not overlap:",
      no_validation_test_overlap)
print("All 1,579 rows assigned:", all_rows_assigned)
print("Every date assigned exactly once:", all_dates_assigned_once)


training_target = train_df["alert_started_tomorrow"]

training_class_counts = training_target.value_counts()

majority_class = training_class_counts.idxmax()
historical_positive_frequency = training_target.mean()

print("\nTraining-only baseline information:")
print("Training majority class:", majority_class)
print(
    "Training historical positive frequency:",
    round(historical_positive_frequency, 4)
)



y_validation = validation_df["alert_started_tomorrow"]

majority_predictions = np.full(
    len(validation_df),
    majority_class,
    dtype=int
)


persistence_predictions = (
    validation_df["alert_started_today"]
    .astype(int)
    .to_numpy()
)


historical_probabilities = np.full(
    len(validation_df),
    historical_positive_frequency,
    dtype=float
)

historical_predictions = (
    historical_probabilities >= 0.5
).astype(int)


def evaluate_baseline(name, y_true, y_pred):
    print(f"\n{name}")

    print(
        "Accuracy:",
        round(accuracy_score(y_true, y_pred), 4)
    )

    print(
        "Balanced accuracy:",
        round(balanced_accuracy_score(y_true, y_pred), 4)
    )

    print(
        "Precision:",
        round(
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            4
        )
    )

    print(
        "Recall:",
        round(
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            4
        )
    )

    print(
        "F1:",
        round(
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            4
        )
    )

    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(matrix)


evaluate_baseline(
    "Majority-class baseline — validation only",
    y_validation,
    majority_predictions
)

evaluate_baseline(
    "Persistence baseline — validation only",
    y_validation,
    persistence_predictions
)

evaluate_baseline(
    "Historical-frequency baseline — validation only",
    y_validation,
    historical_predictions
)


if y_validation.nunique() == 2:
    historical_roc_auc = roc_auc_score(
        y_validation,
        historical_probabilities
    )

    print(
        "\nHistorical-frequency probability ROC-AUC:",
        round(historical_roc_auc, 4)
    )
else:
    print(
        "\nHistorical-frequency ROC-AUC cannot be calculated "
        "because validation contains only one class."
    )

print(
    "\nNo test-set predictions or test performance "
    "were calculated."
)

Dates are strictly increasing: True
Total available observations: 1579

Training period
Start date: 2022-02-26 00:00:00
End date: 2025-03-06 00:00:00
Rows: 1105
No alert tomorrow: 486 (43.98%)
Alert tomorrow: 619 (56.02%)

Validation period
Start date: 2025-03-07 00:00:00
End date: 2025-10-29 00:00:00
Rows: 237
No alert tomorrow: 139 (58.65%)
Alert tomorrow: 98 (41.35%)

Test period
Start date: 2025-10-30 00:00:00
End date: 2026-06-23 00:00:00
Rows: 237
No alert tomorrow: 154 (64.98%)
Alert tomorrow: 83 (35.02%)

Split verification:
Train and validation do not overlap: True
Train and test do not overlap: True
Validation and test do not overlap: True
All 1,579 rows assigned: True
Every date assigned exactly once: True

Training-only baseline information:
Training majority class: 1
Training historical positive frequency: 0.5602

Majority-class baseline — validation only
Accuracy: 0.4135
Balanced accuracy: 0.5
Precision: 0.4135
Recall: 1.0
F1: 0.5851
Confusion matrix [[TN, FP], [FN, TP]]:

In [13]:
import numpy as np
import pandas as pd


feature_work = (
    model_df
    .sort_values("date")
    .reset_index(drop=True)
    .copy()
)

feature_work["alert_count_today"] = feature_work["alert_count"]


feature_work["alert_started_lag_1"] = (
    feature_work["alert_started_today"].shift(1)
)

feature_work["alert_started_lag_2"] = (
    feature_work["alert_started_today"].shift(2)
)

feature_work["alert_started_lag_7"] = (
    feature_work["alert_started_today"].shift(7)
)

feature_work["alert_count_lag_1"] = (
    feature_work["alert_count_today"].shift(1)
)


feature_work["alert_frequency_7d"] = (
    feature_work["alert_started_today"]
    .rolling(window=7, min_periods=7)
    .mean()
)

feature_work["alert_frequency_30d"] = (
    feature_work["alert_started_today"]
    .rolling(window=30, min_periods=30)
    .mean()
)

feature_work["alert_count_sum_7d"] = (
    feature_work["alert_count_today"]
    .rolling(window=7, min_periods=7)
    .sum()
)


last_alert_date = (
    feature_work["date"]
    .where(feature_work["alert_started_today"] == 1)
    .ffill()
)

feature_work["days_since_last_alert"] = (
    feature_work["date"] - last_alert_date
).dt.days


tomorrow_date = feature_work["date"] + pd.Timedelta(days=1)

feature_work["tomorrow_day_of_week"] = tomorrow_date.dt.dayofweek
feature_work["tomorrow_month"] = tomorrow_date.dt.month


rows_before = len(feature_work)

feature_df = feature_work.iloc[29:].copy().reset_index(drop=True)

rows_removed = rows_before - len(feature_df)

feature_columns = [
    "alert_started_today",
    "alert_count_today",
    "alert_started_lag_1",
    "alert_started_lag_2",
    "alert_started_lag_7",
    "alert_count_lag_1",
    "alert_frequency_7d",
    "alert_frequency_30d",
    "alert_count_sum_7d",
    "days_since_last_alert",
    "tomorrow_day_of_week",
    "tomorrow_month"
]

feature_df = feature_df[
    ["date"] + feature_columns + ["alert_started_tomorrow"]
].copy()


integer_columns = [
    "alert_started_lag_1",
    "alert_started_lag_2",
    "alert_started_lag_7",
    "alert_count_lag_1",
    "alert_count_sum_7d",
    "days_since_last_alert"
]

feature_df[integer_columns] = feature_df[integer_columns].astype(int)


print("Final feature columns:")
print(feature_columns)

print("\nRows before feature-history removal:", rows_before)
print("Rows removed because 30-day history was unavailable:", rows_removed)
print("Final feature rows:", len(feature_df))

print("\nDuplicate dates:", feature_df["date"].duplicated().sum())
print("Dates are unique:", feature_df["date"].is_unique)
print("Total missing values:", feature_df.isna().sum().sum())

print("\nMissing values by column:")
print(feature_df.isna().sum())


display_columns = [
    "date",
    "alert_started_today",
    "alert_count_today",
    "alert_started_lag_1",
    "alert_started_lag_2",
    "alert_started_lag_7",
    "alert_count_lag_1",
    "alert_frequency_7d",
    "alert_frequency_30d",
    "alert_count_sum_7d",
    "days_since_last_alert",
    "tomorrow_day_of_week",
    "tomorrow_month",
    "alert_started_tomorrow"
]

print("\nTwelve consecutive completed rows:")
display(feature_df[display_columns].head(12))

check_position = 35

selected_date = feature_work.loc[check_position, "date"]

expected_lag_1 = feature_work.loc[
    check_position - 1,
    "alert_started_today"
]

expected_lag_7 = feature_work.loc[
    check_position - 7,
    "alert_started_today"
]

expected_frequency_7d = (
    feature_work.loc[
        check_position - 6 : check_position,
        "alert_started_today"
    ].mean()
)

expected_count_sum_7d = (
    feature_work.loc[
        check_position - 6 : check_position,
        "alert_count_today"
    ].sum()
)

expected_tomorrow_date = selected_date + pd.Timedelta(days=1)
expected_tomorrow_day_of_week = expected_tomorrow_date.dayofweek
expected_tomorrow_month = expected_tomorrow_date.month

print("\nManual verification for selected row:")
print("Selected date:", selected_date)

print(
    "Lag 1 stored / expected:",
    feature_work.loc[check_position, "alert_started_lag_1"],
    "/",
    expected_lag_1
)

print(
    "Lag 7 stored / expected:",
    feature_work.loc[check_position, "alert_started_lag_7"],
    "/",
    expected_lag_7
)

print(
    "7-day frequency stored / expected:",
    round(feature_work.loc[check_position, "alert_frequency_7d"], 6),
    "/",
    round(expected_frequency_7d, 6)
)

print(
    "7-day count sum stored / expected:",
    feature_work.loc[check_position, "alert_count_sum_7d"],
    "/",
    expected_count_sum_7d
)

print(
    "Tomorrow day of week stored / expected:",
    feature_work.loc[check_position, "tomorrow_day_of_week"],
    "/",
    expected_tomorrow_day_of_week
)

print(
    "Tomorrow month stored / expected:",
    feature_work.loc[check_position, "tomorrow_month"],
    "/",
    expected_tomorrow_month
)

print("\nSeven-day source window used for the selected row:")
display(
    feature_work.loc[
        check_position - 6 : check_position,
        ["date", "alert_started_today", "alert_count_today"]
    ]
)

Final feature columns:
['alert_started_today', 'alert_count_today', 'alert_started_lag_1', 'alert_started_lag_2', 'alert_started_lag_7', 'alert_count_lag_1', 'alert_frequency_7d', 'alert_frequency_30d', 'alert_count_sum_7d', 'days_since_last_alert', 'tomorrow_day_of_week', 'tomorrow_month']

Rows before feature-history removal: 1579
Rows removed because 30-day history was unavailable: 29
Final feature rows: 1550

Duplicate dates: 0
Dates are unique: True
Total missing values: 0

Missing values by column:
date                      0
alert_started_today       0
alert_count_today         0
alert_started_lag_1       0
alert_started_lag_2       0
alert_started_lag_7       0
alert_count_lag_1         0
alert_frequency_7d        0
alert_frequency_30d       0
alert_count_sum_7d        0
days_since_last_alert     0
tomorrow_day_of_week      0
tomorrow_month            0
alert_started_tomorrow    0
dtype: int64

Twelve consecutive completed rows:


,date,alert_started_today,alert_count_today,alert_started_lag_1,alert_started_lag_2,alert_started_lag_7,alert_count_lag_1,alert_frequency_7d,alert_frequency_30d,alert_count_sum_7d,days_since_last_alert,tomorrow_day_of_week,tomorrow_month,alert_started_tomorrow
0,2022-03-27,1,1,1,1,1,2,0.714286,0.833333,14,0,0,3,1
1,2022-03-28,1,1,1,1,1,1,0.714286,0.833333,10,0,1,3,1
2,2022-03-29,1,5,1,1,1,1,0.714286,0.833333,10,0,2,3,1
3,2022-03-30,1,1,1,1,0,5,0.857143,0.833333,11,0,3,3,1
4,2022-03-31,1,1,1,1,0,1,1.000000,0.833333,12,0,4,4,1
5,2022-04-01,1,1,1,1,1,1,1.000000,0.833333,12,0,5,4,1
6,2022-04-02,1,2,1,1,1,1,1.000000,0.833333,12,0,6,4,1
7,2022-04-03,1,4,1,1,1,2,1.000000,0.833333,15,0,0,4,1
8,2022-04-04,1,4,1,1,1,4,1.000000,0.866667,18,0,1,4,1
9,2022-04-05,1,3,1,1,1,4,1.000000,0.866667,16,0,2,4,0



Manual verification for selected row:
Selected date: 2022-04-02 00:00:00
Lag 1 stored / expected: 1.0 / 1
Lag 7 stored / expected: 1.0 / 1
7-day frequency stored / expected: 1.0 / 1.0
7-day count sum stored / expected: 12.0 / 12
Tomorrow day of week stored / expected: 6 / 6
Tomorrow month stored / expected: 4 / 4

Seven-day source window used for the selected row:


,date,alert_started_today,alert_count_today
29,2022-03-27,1,1
30,2022-03-28,1,1
31,2022-03-29,1,5
32,2022-03-30,1,1
33,2022-03-31,1,1
34,2022-04-01,1,1
35,2022-04-02,1,2


In [14]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


numerical_features = [
    "alert_started_today",
    "alert_count_today",
    "alert_started_lag_1",
    "alert_started_lag_2",
    "alert_started_lag_7",
    "alert_count_lag_1",
    "alert_frequency_7d",
    "alert_frequency_30d",
    "alert_count_sum_7d",
    "days_since_last_alert"
]

categorical_features = [
    "tomorrow_day_of_week",
    "tomorrow_month"
]

all_features = numerical_features + categorical_features
target_column = "alert_started_tomorrow"


feature_train_df = feature_df.loc[
    feature_df["date"] <= pd.Timestamp("2025-03-06")
].copy()

feature_validation_df = feature_df.loc[
    feature_df["date"].between(
        pd.Timestamp("2025-03-07"),
        pd.Timestamp("2025-10-29")
    )
].copy()

feature_test_df = feature_df.loc[
    feature_df["date"] >= pd.Timestamp("2025-10-30")
].copy()

def print_split_range(name, period_df):
    print(
        f"{name}: {len(period_df)} rows, "
        f"{period_df['date'].min().date()} to "
        f"{period_df['date'].max().date()}"
    )

print_split_range("Training", feature_train_df)
print_split_range("Validation", feature_validation_df)
print_split_range("Test", feature_test_df)

X_train = feature_train_df[all_features].copy()
y_train = feature_train_df[target_column].copy()

X_validation = feature_validation_df[all_features].copy()
y_validation = feature_validation_df[target_column].copy()

print("\nTraining positive rate:",
      round(y_train.mean() * 100, 2), "%")

print("Validation positive rate:",
      round(y_validation.mean() * 100, 2), "%")


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="drop"
)


logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                random_state=42,
                max_iter=1000
            )
        )
    ]
)

# Fit scaling, encoding and model parameters on training data only
logistic_pipeline.fit(X_train, y_train)


validation_probabilities = logistic_pipeline.predict_proba(
    X_validation
)[:, 1]

validation_predictions = (
    validation_probabilities >= 0.5
).astype(int)


logistic_results = {
    "accuracy": accuracy_score(
        y_validation,
        validation_predictions
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_validation,
        validation_predictions
    ),
    "precision": precision_score(
        y_validation,
        validation_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_validation,
        validation_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_validation,
        validation_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_validation,
        validation_probabilities
    )
}

validation_confusion_matrix = confusion_matrix(
    y_validation,
    validation_predictions,
    labels=[0, 1]
)

print("\nLogistic Regression — validation only")

for metric_name, metric_value in logistic_results.items():
    print(f"{metric_name}: {metric_value:.4f}")

print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(validation_confusion_matrix)


persistence_results = {
    "accuracy": 0.5359,
    "balanced_accuracy": 0.5215,
    "precision": 0.4388,
    "recall": 0.4388,
    "f1": 0.4388
}

comparison_rows = []

for metric_name in persistence_results:
    comparison_rows.append({
        "metric": metric_name,
        "persistence_baseline": persistence_results[metric_name],
        "logistic_regression": logistic_results[metric_name],
        "logistic_minus_persistence": (
            logistic_results[metric_name]
            - persistence_results[metric_name]
        )
    })

comparison_df = pd.DataFrame(comparison_rows)

print("\nValidation comparison:")
display(comparison_df.round(4))

fitted_preprocessor = logistic_pipeline.named_steps["preprocessor"]
fitted_model = logistic_pipeline.named_steps["model"]

transformed_feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

coefficient_df = pd.DataFrame({
    "transformed_feature": transformed_feature_names,
    "coefficient": fitted_model.coef_[0]
})

coefficient_df["absolute_coefficient"] = (
    coefficient_df["coefficient"].abs()
)

coefficient_df = (
    coefficient_df
    .sort_values("absolute_coefficient", ascending=False)
    .reset_index(drop=True)
)

print("\nTransformed feature names:")
print(transformed_feature_names.tolist())

print("\nLogistic Regression coefficients:")
display(
    coefficient_df[
        ["transformed_feature", "coefficient"]
    ]
)

print("\nModel intercept:")
print(fitted_model.intercept_[0])

print(
    "\nTest data was used only to confirm its row count "
    "and date range. No test targets, predictions or metrics "
    "were inspected."
)

Training: 1076 rows, 2022-03-27 to 2025-03-06
Validation: 237 rows, 2025-03-07 to 2025-10-29
Test: 237 rows, 2025-10-30 to 2026-06-23

Training positive rate: 55.3 %
Validation positive rate: 41.35 %

Logistic Regression — validation only
accuracy: 0.5612
balanced_accuracy: 0.5356
precision: 0.4634
recall: 0.3878
f1: 0.4222
roc_auc: 0.5350

Confusion matrix [[TN, FP], [FN, TP]]:
[[95 44]
 [60 38]]

Validation comparison:


,metric,persistence_baseline,logistic_regression,logistic_minus_persistence
0,accuracy,0.5359,0.5612,0.0253
1,balanced_accuracy,0.5215,0.5356,0.0141
2,precision,0.4388,0.4634,0.0246
3,recall,0.4388,0.3878,-0.0510
4,f1,0.4388,0.4222,-0.0166



Transformed feature names:
['numerical__alert_started_today', 'numerical__alert_count_today', 'numerical__alert_started_lag_1', 'numerical__alert_started_lag_2', 'numerical__alert_started_lag_7', 'numerical__alert_count_lag_1', 'numerical__alert_frequency_7d', 'numerical__alert_frequency_30d', 'numerical__alert_count_sum_7d', 'numerical__days_since_last_alert', 'categorical__tomorrow_day_of_week_0', 'categorical__tomorrow_day_of_week_1', 'categorical__tomorrow_day_of_week_2', 'categorical__tomorrow_day_of_week_3', 'categorical__tomorrow_day_of_week_4', 'categorical__tomorrow_day_of_week_5', 'categorical__tomorrow_day_of_week_6', 'categorical__tomorrow_month_1', 'categorical__tomorrow_month_2', 'categorical__tomorrow_month_3', 'categorical__tomorrow_month_4', 'categorical__tomorrow_month_5', 'categorical__tomorrow_month_6', 'categorical__tomorrow_month_7', 'categorical__tomorrow_month_8', 'categorical__tomorrow_month_9', 'categorical__tomorrow_month_10', 'categorical__tomorrow_month_11

,transformed_feature,coefficient
0,categorical__tomorrow_month_12,0.372412
1,categorical__tomorrow_day_of_week_4,0.293133
2,categorical__tomorrow_month_9,-0.284927
3,categorical__tomorrow_day_of_week_6,-0.283999
4,numerical__alert_frequency_30d,0.260879
5,categorical__tomorrow_month_2,-0.214564
6,categorical__tomorrow_day_of_week_5,-0.213164
7,categorical__tomorrow_month_4,-0.199978
8,categorical__tomorrow_day_of_week_2,0.196683
9,categorical__tomorrow_month_10,0.191289



Model intercept:
0.1913795978259355

Test data was used only to confirm its row count and date range. No test targets, predictions or metrics were inspected.


In [15]:
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


development_df = (
    pd.concat(
        [feature_train_df, feature_validation_df],
        axis=0
    )
    .sort_values("date")
    .reset_index(drop=True)
    .copy()
)

X_development = development_df[all_features].copy()
y_development = development_df[target_column].copy()

X_test = feature_test_df[all_features].copy()
y_test = feature_test_df[target_column].copy()

print(
    "Development period:",
    development_df["date"].min().date(),
    "to",
    development_df["date"].max().date(),
    "| Rows:",
    len(development_df)
)

print(
    "Test period:",
    feature_test_df["date"].min().date(),
    "to",
    feature_test_df["date"].max().date(),
    "| Rows:",
    len(feature_test_df)
)

final_pipeline = clone(logistic_pipeline)

final_pipeline.fit(
    X_development,
    y_development
)

test_probabilities = final_pipeline.predict_proba(
    X_test
)[:, 1]

test_logistic_predictions = (
    test_probabilities >= 0.5
).astype(int)

logistic_test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        test_logistic_predictions
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_test,
        test_logistic_predictions
    ),
    "precision": precision_score(
        y_test,
        test_logistic_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        test_logistic_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_test,
        test_logistic_predictions,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        test_probabilities
    )
}

logistic_test_confusion = confusion_matrix(
    y_test,
    test_logistic_predictions,
    labels=[0, 1]
)


test_persistence_predictions = (
    feature_test_df["alert_started_today"]
    .astype(int)
    .to_numpy()
)

persistence_test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        test_persistence_predictions
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_test,
        test_persistence_predictions
    ),
    "precision": precision_score(
        y_test,
        test_persistence_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        test_persistence_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_test,
        test_persistence_predictions,
        zero_division=0
    )
}

persistence_test_confusion = confusion_matrix(
    y_test,
    test_persistence_predictions,
    labels=[0, 1]
)

print("\nLogistic Regression — final test")

for metric_name, metric_value in logistic_test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(logistic_test_confusion)

print("\nPersistence baseline — final test")

for metric_name, metric_value in persistence_test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(persistence_test_confusion)


comparison_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1"
]

final_comparison_df = pd.DataFrame({
    "metric": comparison_metrics,
    "persistence_baseline": [
        persistence_test_metrics[metric]
        for metric in comparison_metrics
    ],
    "logistic_regression": [
        logistic_test_metrics[metric]
        for metric in comparison_metrics
    ]
})

final_comparison_df["logistic_minus_persistence"] = (
    final_comparison_df["logistic_regression"]
    - final_comparison_df["persistence_baseline"]
)


roc_row = pd.DataFrame({
    "metric": ["roc_auc"],
    "persistence_baseline": [np.nan],
    "logistic_regression": [
        logistic_test_metrics["roc_auc"]
    ],
    "logistic_minus_persistence": [np.nan]
})

final_comparison_df = pd.concat(
    [final_comparison_df, roc_row],
    ignore_index=True
)

print("\nFinal test comparison:")
display(final_comparison_df.round(4))


final_comparison_df.to_csv(
    "final_test_metrics.csv",
    index=False
)

joblib.dump(
    final_pipeline,
    "logistic_regression_pipeline.joblib"
)

print("\nSaved files:")
print("final_test_metrics.csv")
print("logistic_regression_pipeline.joblib")


logistic_balanced = logistic_test_metrics[
    "balanced_accuracy"
]

persistence_balanced = persistence_test_metrics[
    "balanced_accuracy"
]

if logistic_balanced > persistence_balanced:
    print(
        "\nConclusion: Logistic Regression beats persistence "
        "on test balanced accuracy."
    )
elif logistic_balanced < persistence_balanced:
    print(
        "\nConclusion: Logistic Regression does not beat "
        "persistence on test balanced accuracy."
    )
else:
    print(
        "\nConclusion: Logistic Regression and persistence "
        "have equal test balanced accuracy."
    )

if logistic_test_metrics["roc_auc"] >= 0.70:
    signal_description = "relatively strong"
elif logistic_test_metrics["roc_auc"] >= 0.60:
    signal_description = "moderate"
elif logistic_test_metrics["roc_auc"] > 0.50:
    signal_description = "weak"
elif logistic_test_metrics["roc_auc"] == 0.50:
    signal_description = "no better than random ranking"
else:
    signal_description = "weaker than random ranking"

print(
    "Based on test ROC-AUC, the model's predictive signal is",
    signal_description + "."
)

print(
    "\nThis was the single final test evaluation. "
    "No further model, feature, threshold or setting changes "
    "should be made in response to these results."
)

Development period: 2022-03-27 to 2025-10-29 | Rows: 1313
Test period: 2025-10-30 to 2026-06-23 | Rows: 237

Logistic Regression — final test
accuracy: 0.5907
balanced_accuracy: 0.5073
precision: 0.3654
recall: 0.2289
f1: 0.2815
roc_auc: 0.4979
Confusion matrix [[TN, FP], [FN, TP]]:
[[121  33]
 [ 64  19]]

Persistence baseline — final test
accuracy: 0.5654
balanced_accuracy: 0.5239
precision: 0.3810
recall: 0.3855
f1: 0.3832
Confusion matrix [[TN, FP], [FN, TP]]:
[[102  52]
 [ 51  32]]

Final test comparison:


,metric,persistence_baseline,logistic_regression,logistic_minus_persistence
0,accuracy,0.5654,0.5907,0.0253
1,balanced_accuracy,0.5239,0.5073,-0.0166
2,precision,0.3810,0.3654,-0.0156
3,recall,0.3855,0.2289,-0.1566
4,f1,0.3832,0.2815,-0.1018
5,roc_auc,NaN,0.4979,NaN



Saved files:
final_test_metrics.csv
logistic_regression_pipeline.joblib

Conclusion: Logistic Regression does not beat persistence on test balanced accuracy.
Based on test ROC-AUC, the model's predictive signal is weaker than random ranking.

This was the single final test evaluation. No further model, feature, threshold or setting changes should be made in response to these results.


In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

figures_dir = "figures"
os.makedirs(figures_dir, exist_ok=True)

saved_files = []

monthly_df = daily_df.copy()
monthly_df["month"] = monthly_df["date"].dt.to_period("M")

monthly_alert_percentage = (
    monthly_df
    .groupby("month")["alert_started_today"]
    .mean()
    .mul(100)
)

monthly_dates = monthly_alert_percentage.index.to_timestamp()

plt.figure(figsize=(12, 5))
plt.plot(
    monthly_dates,
    monthly_alert_percentage.values,
    marker="o",
    markersize=3,
    linewidth=1.5
)
plt.title(
    "Monthly Percentage of Days with at Least One Air-Raid Alert Start\n"
    "Khmelnytska Oblast"
)
plt.xlabel("Month")
plt.ylabel("Alert Days (%)")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

filename = os.path.join(
    figures_dir,
    "monthly_alert_day_percentage.png"
)
plt.savefig(filename, dpi=300, bbox_inches="tight")
plt.close()
saved_files.append(filename)

rolling_30d = (
    daily_df["alert_started_today"]
    .rolling(window=30, min_periods=30)
    .mean()
    .mul(100)
)

plt.figure(figsize=(12, 5))
plt.plot(
    daily_df["date"],
    rolling_30d,
    linewidth=1.5
)
plt.title(
    "30-Day Rolling Alert-Day Frequency\n"
    "Khmelnytska Oblast"
)
plt.xlabel("Date")
plt.ylabel("Alert Days in Previous 30 Days (%)")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

filename = os.path.join(
    figures_dir,
    "rolling_30_day_alert_frequency.png"
)
plt.savefig(filename, dpi=300, bbox_inches="tight")
plt.close()
saved_files.append(filename)

metrics_to_plot = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1"
]

metric_labels = [
    "Accuracy",
    "Balanced\nAccuracy",
    "Precision",
    "Recall",
    "F1"
]

def save_metric_comparison(
    results_df,
    title,
    output_filename
):
    plot_df = (
        results_df
        .set_index("metric")
        .loc[metrics_to_plot]
    )

    x_positions = np.arange(len(metrics_to_plot))
    bar_width = 0.36

    plt.figure(figsize=(10, 5))

    plt.bar(
        x_positions - bar_width / 2,
        plot_df["persistence_baseline"],
        width=bar_width,
        label="Persistence"
    )

    plt.bar(
        x_positions + bar_width / 2,
        plot_df["logistic_regression"],
        width=bar_width,
        label="Logistic Regression"
    )

    plt.title(title)
    plt.xlabel("Metric")
    plt.ylabel("Score")
    plt.xticks(x_positions, metric_labels)
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    path = os.path.join(figures_dir, output_filename)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

    saved_files.append(path)

save_metric_comparison(
    results_df=comparison_df,
    title=(
        "Validation Performance: Logistic Regression "
        "vs Persistence"
    ),
    output_filename="validation_model_comparison.png"
)


save_metric_comparison(
    results_df=final_comparison_df,
    title=(
        "Final Test Performance: Logistic Regression "
        "vs Persistence"
    ),
    output_filename="final_test_model_comparison.png"
)

def save_confusion_matrix(
    matrix,
    title,
    output_filename
):
    plt.figure(figsize=(5, 4))

    plt.imshow(matrix)
    plt.title(title)
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")

    plt.xticks([0, 1], ["No Alert", "Alert"])
    plt.yticks([0, 1], ["No Alert", "Alert"])

    threshold = matrix.max() / 2

    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix[row, column]

            plt.text(
                column,
                row,
                str(value),
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
                fontsize=12
            )

    plt.colorbar(label="Number of Days")
    plt.tight_layout()

    path = os.path.join(figures_dir, output_filename)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

    saved_files.append(path)


save_confusion_matrix(
    matrix=logistic_test_confusion,
    title="Final Test Confusion Matrix: Logistic Regression",
    output_filename="test_confusion_matrix_logistic_regression.png"
)

save_confusion_matrix(
    matrix=persistence_test_confusion,
    title="Final Test Confusion Matrix: Persistence Baseline",
    output_filename="test_confusion_matrix_persistence.png"
)

print("Saved figure files:")

for path in saved_files:
    print(path)

Saved figure files:
figures/monthly_alert_day_percentage.png
figures/rolling_30_day_alert_frequency.png
figures/validation_model_comparison.png
figures/final_test_model_comparison.png
figures/test_confusion_matrix_logistic_regression.png
figures/test_confusion_matrix_persistence.png


In [19]:
import shutil
from google.colab import files

# Create one ZIP archive with all figures
shutil.make_archive(
    "figures",
    "zip",
    "figures"
)

files.download("figures.zip")
files.download("final_test_metrics.csv")
files.download("logistic_regression_pipeline.joblib")
files.download("volunteer_data_en_snapshot_2026-06-25.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>